# Actividad 3

## Instrucciones

- Investigar objetos del pipeline de sklearn para calibrar hiperparámetros de un modelo de Machine Learning
- Mostrar el diagrama que se genera del pipeline (se puede usar un Jupyter Notebook para este apartado)
- Empaquetar pipeline (setup.py, toml, etc) para poder compartir con compañeros del equipo
- Mostrar con una captura de pantalla que se pudo ejecutar el pipeline en diferentes computadoras
- Usar como referencia las etapas del ciclo de CRISP-DM relacionadas a diseño y evaluación de modelos de Machine Learning (Modeling, Evaluation)
- La actividad cuenta como asistencia para la clase virtual del viernes 14 de agosto 2026, es una entrega individual

## CRISP-DM: Progreso de la Actividad

| Fase | Estado | Descripción |
|------|--------|-------------|
| **Business Understanding** | [OK] | Predicción de límite de crédito promedio por cliente |
| **Data Understanding** | [OK] | 660 registros, 7 variables, tipología explorada |
| **Data Preparation** | [OK] | Extracción, filtrado, preprocesamiento completado |
| **Modeling** | [PROGRESO] | Selección e implementación de Linear Regression |
| **Evaluation** | [PENDIENTE] | Métricas y validación del modelo |
| **Deployment** | [PENDIENTE] | Empaquetado (setup.py) para compartir |

## Selección del Modelo

**Linear Regression**: Pocas variables (4 después de preprocesamiento) requieren interpretabilidad directa de coeficientes sobre límite de crédito.

## Hiperparámetros a Calibrar

- `fit_intercept=True`: Permitir término independiente $\beta_0$ en la ecuación de regresión.
- `positive=False`: Sin restricción de signos en coeficientes (relaciones pueden ser negativas o positivas).

## Librerias

El pipeline se importa del paquete `ml_pipeline`, instalado con `pip install -e .`

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import pandas as pd

# Funciones del paquete instalado con: pip install -e .
from ml_pipeline import (
    FeatureExtractor,
    RowFilter,
    crear_preprocesador,
    crear_pipeline_modelo,
    calibrar_modelo,
    evaluar_modelo,
    analizar_overfitting,
    mostrar_coeficientes,
    validacion_cruzada,
)

## Descripción del Dataset

- Sl_No: (Según google Serial Number) Es un número de serie, sirve para indicar el número de identificiación o la posición consecutiva de un elemento

- Customer Key: Es un id  único para cada cliente

- Avg_Credit_Limit: Es el límite de crédito promedio asignado a un cliente

- Total_Credit_Cards: Es la cantidad total de tarjetas de crédito que un cliente posee

- Total_visits_online: Es la frecuencia con la que el cliente se conecta a la banca en línea

- Total_calls_made: Es la cantidad de llamadas que el cliente ha realiado al soporte del banco

In [40]:
file_id = '10S8JVFiLfCoRB9mb0NJG5YhITyXbH37B'

download_url = f'https://drive.google.com/uc?export=download&id={file_id}'

data = pd.read_csv(download_url)

In [ ]:
print("\nTipos de datos:")
print(data.dtypes)

In [42]:
X = data.drop(columns=["Sl_No", "Customer Key"])

## Extraccion

In [ ]:
# FeatureExtractor viene del paquete ml_pipeline.transformers
FeatureExtractor??

## Filtrado

In [ ]:
# RowFilter viene del paquete ml_pipeline.transformers
RowFilter??

## Limpieza

In [ ]:
columnas_a_descartar = ["Sl_No", "Customer Key"]
rangos_numericos = {}

cleaning_pipeline = Pipeline(steps=[
    ('extraccion', FeatureExtractor(columns_to_drop=columnas_a_descartar)),
    ('filtrado', RowFilter(max_null_ratio=0.5, numeric_bounds=rangos_numericos)),
])

data_clean = cleaning_pipeline.fit_transform(data)
print('Shape original:', data.shape, '-> Shape limpio:', data_clean.shape)

## Split de Datos

In [ ]:
target_column = 'Avg_Credit_Limit'
X = data_clean.drop(columns=[target_column])
y = data_clean[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('X_train:', X_train.shape, ' X_test:', X_test.shape)

## Transformacion

In [ ]:
preprocessor = crear_preprocesador()
preprocessor

## Pipeline Final

In [ ]:
model_pipeline = Pipeline(steps=[
    ('preprocesamiento', preprocessor)
])

X_train_transformed = model_pipeline.fit_transform(X_train)
X_test_transformed = model_pipeline.transform(X_test)

print('X_train_transformed:', X_train_transformed.shape)
print('X_test_transformed:', X_test_transformed.shape)

## Diagrama

In [ ]:
# Diagrama completo: Preprocesamiento + Modelo
# Crear pipeline con preprocesamiento + Linear Regression

full_pipeline = Pipeline(steps=[
    ('extraccion', FeatureExtractor(columns_to_drop=columnas_a_descartar)),
    ('filtrado', RowFilter(max_null_ratio=0.5, numeric_bounds=rangos_numericos)),
    ('preprocesamiento', preprocessor),
    ('modelo', LinearRegression())
])

print("PIPELINE COMPLETO: PREPROCESAMIENTO + MODELO LINEAR REGRESSION")
print("-" * 70)
full_pipeline

## Modeling: Implementacion de Linear Regression

Calibracion de hiperparametros con GridSearchCV sobre 5 folds.

In [ ]:
lr_pipeline = crear_pipeline_modelo(preprocessor)

param_grid = {
    'modelo__fit_intercept': [True, False],
    'modelo__positive': [True, False]
}

grid_search = calibrar_modelo(lr_pipeline, X_train, y_train, param_grid, cv=5)

print("GRID SEARCH: CALIBRACION DE HIPERPARAMETROS")
print("-" * 60)
print(f"Mejores hiperparametros: {grid_search.best_params_}")
print(f"Mejor R_2 (CV): {grid_search.best_score_:.4f}")

In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)[
    ['param_modelo__fit_intercept', 'param_modelo__positive', 'mean_test_score', 'std_test_score']
]
results_df.columns = ['fit_intercept', 'positive', 'R_2_mean', 'R_2_std']
print("\nTodos los parametros evaluados:")
print(results_df.to_string(index=False))

## Evaluation: Validacion del Modelo

Metricas de desempenio, overfitting, coeficientes y validacion cruzada.

In [ ]:
best_model = grid_search.best_estimator_
metricas = evaluar_modelo(best_model, X_train, y_train, X_test, y_test)

print("EVALUACION DEL MODELO")
print("-" * 60)
print(f"R_2 Train: {metricas['train']['r2']:.4f}")
print(f"R_2 Test:  {metricas['test']['r2']:.4f}")
print(f"MAE Test:  ${metricas['test']['mae']:.2f}")
print(f"RMSE Test: ${metricas['test']['rmse']:.2f}")

In [ ]:
analizar_overfitting(metricas)

In [ ]:
mostrar_coeficientes(best_model)

In [ ]:
validacion_cruzada(best_model, X_train, y_train)

## CRISP-DM: Estado Final

| Fase | Estado | Descripcion |
|------|--------|-------------|
| **Business Understanding** | [OK] | Prediccion de limite de credito por cliente |
| **Data Understanding** | [OK] | 660 registros, 7 variables analizadas |
| **Data Preparation** | [OK] | Extraccion, filtrado y preprocesamiento aplicados |
| **Modeling** | [OK] | Linear Regression calibrado con GridSearchCV |
| **Evaluation** | [OK] | R_2 = 0.5664 en test, overfitting moderado |
| **Deployment** | [OK] | Paquete instalable con pip install -e . |